# Nettoyage des données DVF - Ille-et-Vilaine (2021-2025)

Ce notebook charge les fichiers DVF bruts (un par année), les assemble, filtre les colonnes et les types de biens pertinents, traite les valeurs manquantes et aberrantes, calcule le prix au m², puis exporte un fichier propre dans `data/cleaned/`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Liste des années couvertes par l'analyse
annees = [2021, 2022, 2023, 2024, 2025]

# Liste vide qui va accueillir un DataFrame par année
dataframes = []

for annee in annees:
    chemin = f"../data/raw/dvf_35_{annee}.csv.gz"
    df_annee = pd.read_csv(chemin, low_memory=False)
    df_annee["annee"] = annee
    dataframes.append(df_annee)

df = pd.concat(dataframes, ignore_index=True)

print(df.shape)
df.head()

(337019, 41)


,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude,annee
0,2021-591508,2021-01-07,1,Vente,160000.0,7.0,NaN,RUE BEAUGEARD LANCELOT,0750,35700.0,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,-1.656551,48.119053,2021
1,2021-591508,2021-01-07,1,Vente,160000.0,7.0,NaN,RUE BEAUGEARD LANCELOT,0750,35700.0,...,55.0,3.0,NaN,NaN,NaN,NaN,NaN,-1.656551,48.119053,2021
2,2021-591509,2021-01-06,1,Vente,225000.0,6.0,NaN,BD DES METAIRIES,0432,35510.0,...,81.0,4.0,NaN,NaN,NaN,NaN,NaN,-1.599096,48.121689,2021
3,2021-591510,2021-01-07,1,Vente,304000.0,12.0,NaN,ALL EMILE GERNIGON,0963,35136.0,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,-1.699642,48.091458,2021
4,2021-591510,2021-01-07,1,Vente,304000.0,12.0,NaN,ALL EMILE GERNIGON,0963,35136.0,...,96.0,4.0,NaN,NaN,NaN,NaN,NaN,-1.699642,48.091458,2021


## Exploration initiale

Avant de nettoyer, on regarde d'abord ce que contient réellement le tableau : les types de colonnes, les valeurs manquantes, et les doublons potentiels.

In [3]:
# J'affiche la liste des colonnes de mon tableau, leur nom, le nombre 
# de valeurs non vide qu'elles contiennent et leur type

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 337019 entries, 0 to 337018
Data columns (total 41 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id_mutation                   337019 non-null  str    
 1   date_mutation                 337019 non-null  str    
 2   numero_disposition            337019 non-null  int64  
 3   nature_mutation               337019 non-null  str    
 4   valeur_fonciere               333524 non-null  float64
 5   adresse_numero                216238 non-null  float64
 6   adresse_suffixe               13028 non-null   str    
 7   adresse_nom_voie              332070 non-null  str    
 8   adresse_code_voie             332121 non-null  str    
 9   code_postal                   332118 non-null  float64
 10  code_commune                  337019 non-null  int64  
 11  nom_commune                   337019 non-null  str    
 12  code_departement              337019 non-null  int64  


In [4]:
colonnes_utiles = [
    "date_mutation", "annee", "nature_mutation", "valeur_fonciere",
    "surface_reelle_bati", "type_local", "code_commune", "nom_commune",
    "longitude", "latitude"
]

df[colonnes_utiles].isnull().sum()

date_mutation               0
annee                       0
nature_mutation             0
valeur_fonciere          3495
surface_reelle_bati    232562
type_local             157570
code_commune                0
nom_commune                 0
longitude                8494
latitude                 8494
dtype: int64

In [5]:
df[df["type_local"].isnull()][["nature_mutation", "code_nature_culture", "nature_culture"]].head(10)

,nature_mutation,code_nature_culture,nature_culture
8,Vente en l'état futur d'achèvement,NaN,NaN
9,Vente en l'état futur d'achèvement,NaN,NaN
10,Vente en l'état futur d'achèvement,NaN,NaN
17,Vente,S,sols
21,Vente en l'état futur d'achèvement,NaN,NaN
22,Vente en l'état futur d'achèvement,NaN,NaN
37,Vente,S,sols
39,Vente en l'état futur d'achèvement,NaN,NaN
40,Vente en l'état futur d'achèvement,NaN,NaN
41,Vente,S,sols


In [6]:
df["nature_mutation"].value_counts()

nature_mutation
Vente                                 300950
Vente en l'état futur d'achèvement     28436
Echange                                 4167
Vente terrain à bâtir                   3113
Adjudication                             338
Expropriation                             15
Name: count, dtype: int64

In [7]:
vefa = df[df["nature_mutation"] == "Vente en l'état futur d'achèvement"]

print("Nombre total de lignes VEFA :", len(vefa))
print("Dont surface renseignée :", vefa["surface_reelle_bati"].notnull().sum())
print("Dont type_local renseigné :", vefa["type_local"].notnull().sum())

Nombre total de lignes VEFA : 28436
Dont surface renseignée : 224
Dont type_local renseigné : 491


In [8]:
df = df[df["nature_mutation"] == "Vente"]

print(df.shape)

(300950, 41)


In [9]:
colonnes_utiles = [
    "id_mutation", "id_parcelle", "date_mutation", "annee", "valeur_fonciere",
    "surface_reelle_bati", "type_local", "code_commune", "nom_commune",
    "longitude", "latitude"
]

df = df[colonnes_utiles]

print(df.shape)
df.head()

(300950, 11)


,id_mutation,id_parcelle,date_mutation,annee,valeur_fonciere,surface_reelle_bati,type_local,code_commune,nom_commune,longitude,latitude
0,2021-591508,35238000AZ0165,2021-01-07,2021,160000.0,NaN,Dépendance,35238,Rennes,-1.656551,48.119053
1,2021-591508,35238000AZ0165,2021-01-07,2021,160000.0,55.0,Appartement,35238,Rennes,-1.656551,48.119053
2,2021-591509,35051000AD0139,2021-01-06,2021,225000.0,81.0,Appartement,35051,Cesson-Sévigné,-1.599096,48.121689
3,2021-591510,35281000AC0391,2021-01-07,2021,304000.0,NaN,Dépendance,35281,Saint-Jacques-de-la-Lande,-1.699642,48.091458
4,2021-591510,35281000AC0391,2021-01-07,2021,304000.0,96.0,Maison,35281,Saint-Jacques-de-la-Lande,-1.699642,48.091458


In [10]:
df["type_local"].value_counts(dropna=False)

type_local
NaN                                         122696
Dépendance                                   73693
Maison                                       58061
Appartement                                  37004
Local industriel. commercial ou assimilé      9496
Name: count, dtype: int64

In [11]:
df = df[df["type_local"].isin(["Maison", "Appartement"])]

print(df.shape)
df["type_local"].value_counts()

(95065, 11)


type_local
Maison         58061
Appartement    37004
Name: count, dtype: int64

In [12]:
doublons = df["id_mutation"].duplicated().sum()
print(doublons)

15630


In [13]:
id_dupliques = df[df["id_mutation"].duplicated(keep=False)]["id_mutation"].unique()[:3]

df[df["id_mutation"].isin(id_dupliques)].sort_values("id_mutation")

,id_mutation,id_parcelle,date_mutation,annee,valeur_fonciere,surface_reelle_bati,type_local,code_commune,nom_commune,longitude,latitude
19,2021-591519,35238000BN0154,2021-01-04,2021,562000.0,63.0,Appartement,35238,Rennes,-1.656340,48.111765
20,2021-591519,35238000BN0154,2021-01-04,2021,562000.0,63.0,Appartement,35238,Rennes,-1.656340,48.111765
25,2021-591523,35206073AT0181,2021-01-04,2021,230000.0,86.0,Maison,35206,Noyal-Châtillon-sur-Seiche,-1.666735,48.035223
26,2021-591523,35206073AT0181,2021-01-04,2021,230000.0,98.0,Maison,35206,Noyal-Châtillon-sur-Seiche,-1.666735,48.035223
27,2021-591523,35206073AT0181,2021-01-04,2021,230000.0,98.0,Maison,35206,Noyal-Châtillon-sur-Seiche,-1.666735,48.035223
28,2021-591523,35206073AT0181,2021-01-04,2021,230000.0,100.0,Maison,35206,Noyal-Châtillon-sur-Seiche,-1.666735,48.035223
29,2021-591523,35206073AT0181,2021-01-04,2021,230000.0,86.0,Maison,35206,Noyal-Châtillon-sur-Seiche,-1.666735,48.035223
62,2021-591541,35206073AT0194,2021-01-18,2021,212000.0,100.0,Maison,35206,Noyal-Châtillon-sur-Seiche,-1.665962,48.034903
63,2021-591541,35206073AT0194,2021-01-18,2021,212000.0,98.0,Maison,35206,Noyal-Châtillon-sur-Seiche,-1.665962,48.034903
64,2021-591541,35206073AT0194,2021-01-18,2021,212000.0,98.0,Maison,35206,Noyal-Châtillon-sur-Seiche,-1.665962,48.034903


In [14]:
doublons_exacts = df.duplicated().sum()
print("Lignes strictement identiques (doublons exacts) :", doublons_exacts)

Lignes strictement identiques (doublons exacts) : 11156


In [15]:
df = df.drop_duplicates()

print(df.shape)

(83909, 11)


In [16]:
doublons_restants = df["id_mutation"].duplicated().sum()
print(doublons_restants)

4474


In [17]:
df[df["id_mutation"] == "2021-591523"]

,id_mutation,id_parcelle,date_mutation,annee,valeur_fonciere,surface_reelle_bati,type_local,code_commune,nom_commune,longitude,latitude
25,2021-591523,35206073AT0181,2021-01-04,2021,230000.0,86.0,Maison,35206,Noyal-Châtillon-sur-Seiche,-1.666735,48.035223
26,2021-591523,35206073AT0181,2021-01-04,2021,230000.0,98.0,Maison,35206,Noyal-Châtillon-sur-Seiche,-1.666735,48.035223
28,2021-591523,35206073AT0181,2021-01-04,2021,230000.0,100.0,Maison,35206,Noyal-Châtillon-sur-Seiche,-1.666735,48.035223


In [18]:
agg_rules = {
    "date_mutation": "first",
    "annee": "first",
    "valeur_fonciere": "first",
    "surface_reelle_bati": "sum",
    "type_local": "first",
    "code_commune": "first",
    "nom_commune": "first",
    "longitude": "first",
    "latitude": "first",
}

df = df.groupby("id_mutation").agg(agg_rules).reset_index()

print(df.shape)

(79435, 10)


In [19]:
df[["valeur_fonciere", "surface_reelle_bati"]].describe()

,valeur_fonciere,surface_reelle_bati
count,7.933800e+04,79435.000000
mean,2.484018e+05,88.713048
std,2.286331e+05,47.762683
min,1.000000e+00,1.000000
25%,1.400000e+05,59.000000
50%,2.050000e+05,81.000000
75%,2.978000e+05,111.000000
max,1.484000e+07,1360.000000
